In [9]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml.paths import EXPERIMENT_RESULTS_FILE, EXPERIMENT_CONFIGS_FILE, EXPERIMENT_FEATURE_EFFECT, SELECTION_RESULTS_FILE, SELECTION_CONFIGS_FILE, SELECTION_FEATURE_EFFECT, TRAIN_PATH, TEST_PATH
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 


from ast import For
from cmath import exp

from titanic_ml.common.experiments.runner import run_experiments, run_experiment_group_workflow
from titanic_ml.common.experiments.configurations.exp_config import ALL_EXPERIMENTS
from titanic_ml.common.experiments.configurations.configuration import ALL_FE_CONFIG, ALL_FS_CONFIGS, LOGREG_FS_CONFIGS
from titanic_ml.common.experiments.report import experiment_report, experiment_group_summary_report, baseline_summary_to_markdown, workflow_report
from titanic_ml.common.experiments.save_load import save_results, load_results, save_configs, load_configs, save_feature_effects, load_feature_effects
from titanic_ml.common.experiments.compare import leaderboard, compare_experiment_groups, summarize_group_comparison, titanic_notes_leaderboard, model_progression, analyze_feature_effect, domain_best_by_model_with_baseline_delta, domain_best_by_model, recommended_by_domain_for_model
from titanic_ml.common.models.registry import MODEL_REGISTRY
from titanic_ml.feature_engineering import add_family_features, add_has_cabin, add_title, add_full_title_feature
from titanic_ml.common.data.eda import sample_dataframe

# import inspect
# print(ALL_FS_CONFIGS.__file__)
# print(inspect.getsource(ALL_FS_CONFIGS.overview))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# print(ALL_FS_CONFIGS)
# print()
# print(ALL_EXPERIMENTS)

#### Feature selection pipeline and related functions:

In [11]:
TARGET = "Survived"
results_path = SELECTION_RESULTS_FILE
configs_path = SELECTION_CONFIGS_FILE
feature_effect_path = SELECTION_FEATURE_EFFECT

for experiment_name, exp_config in ALL_FS_CONFIGS.items():
    print(f"Experiment: {experiment_name}")



Experiment: fs__baseline__logreg
Experiment: fs01__age_cb03__logreg
Experiment: fs01__age_cb02__logreg
Experiment: fs02__family_cb07__logreg
Experiment: fs02__family_fe01__logreg
Experiment: fs03__cabin_fe04__logreg
Experiment: fs04__ticket_fe09_batch__logreg
Experiment: fs04__ticket_fe09_fitted__logreg
Experiment: fs04__ticket_fe09_full_context__logreg
Experiment: fs05__fare_fe08__logreg
Experiment: fs05__fare_cb05_fitted__logreg
Experiment: fs05__fare_cb05_full_context__logreg
Experiment: fs06__sex_pclass_fe12__logreg
Experiment: fs06__sex_pclass_cb08__logreg
Experiment: pr01__removing_family__logreg
Experiment: pr02__removing_ticket__logreg
Experiment: pr03__removing_non_raw_fare__logreg
Experiment: pr03__removing_fare__logreg
Experiment: pr03__removing_raw_fare__logreg
Experiment: pr04__removing_age__logreg
Experiment: pr05__removing_cabin__logreg
Experiment: pr06__removing_title__logreg
Experiment: prsc__age_cb02__logreg
Experiment: pr07__removing_sibSp_parch__logreg
Experiment: p

In [12]:
# # Debugging
# for key, values in ALL_FS_CONFIGS["fs__baseline__logreg"]['fs__baseline'].items():
#     print(f"Key: {key}")
#     print(f"Values : {values}")

In [13]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

exp_configs = ALL_FS_CONFIGS["pr05__removing_fe02__decision_tree"]
reference_group = 'pr03__removing_fe09_cb06_using_fe02'
print(exp_configs)
# # Line to rerun all experiments to update the results with the latest code changes. This will take a while.
# # Uncomment to run all experiments and update results.

# for Name, exp_config in ALL_FS_CONFIGS.items():
#     print(f"Running {Name} experiments...")
#     print(f"Experiment config: {exp_config}")
#     group_name = Name.rsplit("__", 1)[0]
#     exp_result = run_experiments(train_df, exp_config, target=TARGET, verbose=True, debug=True, context_df=test_df)
#     result_df = save_results(exp_result, path=results_path)
#     save_configs(exp_config, path=configs_path)
#     if group_name != 'baseline__raw':
#         comparison = compare_experiment_groups(
#             results_df=result_df,
#             reference_group="fs__baseline",
#             compare_groups=[group_name],
#         )
#         feature_effect = analyze_feature_effect(comparison)
#         save_feature_effects(feature_effect, path=feature_effect_path)

{'pr05__removing_fe02': {'pre_cv_feature_pipeline': [], 'pre_cv_scope': None, 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone'], 'onehot_features': ['Sex', 'Embarked', 'Title'], 'ordinal_features': ['Pclass'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'feature_pipeline': [('title', TitleTransformer()), ('family_size', SumColumnsTransformer(constant=1, output_col='FamilySize',
                      source_cols=['SibSp', 'Parch'])), ('is_alone', IsOneTransformer(output_col='IsAlone', source_col='FamilySize'))], 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'stage': 'pr05', 'feature_group': 'removing_fe02', 'group': 'pr05__removing_fe02', 'domain': 'decision_tree', 'tags': set(), 'model_name': 'decision_tree', 'model_params': {'max_depth': 4, 'min_samples_leaf': 5, 'random_state': 42}, 'not

In [14]:
# Work flow for running an experiment group, comparing it to the baseline, and generating a report. 
# This is the main workflow for analyzing the results of an experiment group and generating insights from it.
workflow = run_experiment_group_workflow(
    reference_group=reference_group,
    df=train_df,
    experiment_configs=exp_configs,
    target=TARGET,
    save=True,
    context_df=test_df,
    results_path=results_path,
    configs_path=configs_path,
    feature_effect_path=feature_effect_path,
    debug=True
)
# print("Workflow completed. Here are the results:")
# print("Comparison between baseline and feature engineering group:")
# print(workflow["comparison"])
# print("Summary of comparison:")
# print(workflow["summary"])
# print("Leaderboard:")
# print(workflow["leaderboard"])

Running config: {'pre_cv_feature_pipeline': [], 'pre_cv_scope': None, 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone'], 'onehot_features': ['Sex', 'Embarked', 'Title'], 'ordinal_features': ['Pclass'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'feature_pipeline': [('title', TitleTransformer()), ('family_size', SumColumnsTransformer(constant=1, output_col='FamilySize',
                      source_cols=['SibSp', 'Parch'])), ('is_alone', IsOneTransformer(output_col='IsAlone', source_col='FamilySize'))], 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'stage': 'pr05', 'feature_group': 'removing_fe02', 'group': 'pr05__removing_fe02', 'domain': 'decision_tree', 'tags': set(), 'model_name': 'decision_tree', 'model_params': {'max_depth': 4, 'min_samples_leaf': 5, 'random_state': 42}, 'notes': 'Ex

In [15]:
# Generate a full report for the workflow, including the comparison, summary, and leaderboard. 
# This will be a markdown report that can be easily shared and visualized.
# Mainly used for generating the report for the notebook, but can also be used for generating reports for individual experiment groups or comparisons.
full_report = workflow_report(workflow, top_n=20)
print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])
print()
# For combos:
all_results = load_results(path=results_path)
references = [reference_group]
for reference in references:
    comparison = compare_experiment_groups(
                results_df=all_results,
                reference_group=reference,
                compare_groups=[exp_configs],
            )
    print(f"Comparison summary:")
    print(comparison[["reference_group", "compare_group", "model_name", "test_accuracy_mean_delta", "test_f1_mean_delta"]].to_markdown())
    print()


Full workflow report:

Report
#### pr05__removing_fe02

_Description pending._

<details>
<summary>Conclusion</summary>


##### Interpretation

- Verdict: general_negative
- Discard
- Max delta: -0.003


##### Conclusion

_Conclusion pending._

</details>

<details>
<summary>Experiment details</summary>

##### Comparison vs pr03__removing_fe09_cb06_using_fe02

| reference_group                     | compare_group       | model_name    |   test_accuracy_mean_reference |   test_accuracy_mean_compare |   test_accuracy_mean_delta |   test_f1_mean_reference |   test_f1_mean_compare |   test_f1_mean_delta |
|:------------------------------------|:--------------------|:--------------|-------------------------------:|-----------------------------:|---------------------------:|-------------------------:|-----------------------:|---------------------:|
| pr03__removing_fe09_cb06_using_fe02 | pr05__removing_fe02 | decision_tree |                          0.827 |                        0.824 |    

In [16]:
# raise to quickly run small experiments and reports
raise

RuntimeError: No active exception to reraise

In [ ]:
# Leaderboard without ablations
current_leaderboard = titanic_notes_leaderboard(all_results, top_n=11, selection="exclude")
print("Current leaderboard:")
print(current_leaderboard)

In [ ]:
# for model in MODEL_REGISTRY:
#     model_progression_df = model_progression(workflow["all_results"], model_name=model, metric="test_accuracy_mean")
#     print(f"Model progression for {model}:")
#     print(model_progression_df)
#     print()

#### Feature experiments pipeline and related functions:

In [ ]:
TARGET = "Survived"
ALL_EXPERIMENTS = ALL_FE_CONFIG

results_path = EXPERIMENT_RESULTS_FILE
configs_path = EXPERIMENT_CONFIGS_FILE
feature_effect_path = EXPERIMENT_FEATURE_EFFECT


for experiment_name, exp_config in ALL_EXPERIMENTS.items():
    print(f"Experiment: {experiment_name}")

print(ALL_EXPERIMENTS)


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

exp_configs = ALL_EXPERIMENTS["baseline__raw"]

# # Line to rerun all experiments to update the results with the latest code changes. This will take a while.
# # Uncomment to run all experiments and update results.

# for Name, exp_config in ALL_EXPERIMENTS.items():
#     print(f"Running {Name} experiments...")
#     print(f"Experiment config: {exp_config}")
#     exp_result = run_experiments(train_df, exp_config, target=TARGET, verbose=True, debug=True, context_df=test_df)
#     result_df = save_results(exp_result, path=results_path)
#     save_configs(exp_config, path=configs_path)
#     if Name != 'baseline__raw':
#         comparison = compare_experiment_groups(
#             results_df=result_df,
#             reference_group="baseline__raw",
#             compare_groups=[Name],
#         )
#         feature_effect = analyze_feature_effect(comparison)
#         save_feature_effects(feature_effect, path=feature_effect_path)

print(exp_configs)

In [ ]:
# print("Experiment Configurations:")
# print(exp_configs)
# for exp_config in exp_configs:
#     print(exp_config)

In [ ]:
# Work flow for running an experiment group, comparing it to the baseline, and generating a report. 
# This is the main workflow for analyzing the results of an experiment group and generating insights from it.
workflow = run_experiment_group_workflow(
    df=train_df,
    experiment_configs=exp_configs,
    target=TARGET,
    save=True,
    context_df=test_df,
    results_path=results_path,
    configs_path=configs_path,
    feature_effect_path=feature_effect_path
)
# print("Workflow completed. Here are the results:")
# print("Comparison between baseline and feature engineering group:")
# print(workflow["comparison"])
# print("Summary of comparison:")
# print(workflow["summary"])
# print("Leaderboard:")
# print(workflow["leaderboard"])

In [ ]:
# Generate a full report for the workflow, including the comparison, summary, and leaderboard. 
# This will be a markdown report that can be easily shared and visualized.
# Mainly used for generating the report for the notebook, but can also be used for generating reports for individual experiment groups or comparisons.
full_report = workflow_report(workflow, top_n=20)
print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])
print()
# For combos:
all_results = load_results(path=results_path)
references = ['baseline__raw']
for reference in references:
    comparison = compare_experiment_groups(
                results_df=all_results,
                reference_group=reference,
                compare_groups=[exp_configs],
            )
    print(f"Comparison summary:")
    print(comparison[["reference_group", "compare_group", "model_name", "test_accuracy_mean_delta", "test_f1_mean_delta"]].to_markdown())
    print()


In [ ]:
# raise to quickly run small experiments and reports
raise

In [ ]:
# Leaderboard without ablations
current_leaderboard = titanic_notes_leaderboard(all_results, top_n=100, selection="exclude")
print("Current leaderboard:")
print(current_leaderboard)

In [ ]:
from titanic_ml.common.experiments.inspection import get_feature_pipeline, get_cv_fold, inspect_experiment_fold
from titanic_ml.common.experiments.runner import build_model_pipeline
for exp_name, exp_config in exp_configs.items():
    exp = exp_config
    break

model_pipeline = build_model_pipeline(exp)
for name, step in model_pipeline.steps:
    print(name, type(step).__name__)

In [ ]:
inspection = inspect_experiment_fold(
    df=train_df,
    experiments=ALL_EXPERIMENTS["test_family_ticket_full_context_config"],
    target="Survived",
    context_df=test_df,
    fold=0,
)

In [ ]:
inspection[
    "X_validation_engineered"
].columns

In [ ]:
train = inspection["X_train_cv_input"]
validation = inspection[
    "X_validation_engineered"
]

row = validation.iloc[0]

ticket = row["Ticket"]

print(
    "Ticket:",
    ticket,
)

print(
    "Training count:",
    train["Ticket"].eq(ticket).sum(),
)

print(
    "Engineered count:",
    row["TicketGroupSize"],
)

In [ ]:
validation_raw = inspection[
    "X_train_cv_input"
]

validation = inspection[
    "X_validation_engineered"
]

row = validation.iloc[0]
ticket = row["Ticket"]

print(
    validation_raw["Ticket"]
    .eq(ticket)
    .sum()
)

print(
    row["TicketGroupSize"]
)

In [ ]:
inspection["X_train_cv_input"]
inspection["X_validation_cv_input"]

In [ ]:
df = inspection[
    "X_validation_engineered"
]

(
    df["FamilySize"]
    ==
    df["SibSp"] + df["Parch"] + 1
).all()

In [ ]:
def compare_group_representations(
    df,
    family_col="FamilySize",
    ticket_col="TicketGroupSize",
    family_fare_col="Fare/FamilySize",
    ticket_fare_col="Fare/TicketMember",
):
    same_group_size = df[family_col].eq(df[ticket_col])

    return {
        "group_size_equal_rate": same_group_size.mean(),

        "group_size_mae": (
            df[family_col] - df[ticket_col]
        ).abs().mean(),

        "fare_pearson": df[
            [family_fare_col, ticket_fare_col]
        ].corr(method="pearson").iloc[0, 1],

        "fare_spearman": df[
            [family_fare_col, ticket_fare_col]
        ].corr(method="spearman").iloc[0, 1],

    "family_unique": df[family_fare_col].nunique(),

    "ticket_unique": df[ticket_fare_col].nunique(),
    
    "exact_fare_equal_rate": (
        df[family_fare_col] == df[ticket_fare_col]
    ).mean(),
}


In [ ]:
print("Comparing family and ticket group representations:")
compare_group = compare_group_representations(inspection["X_validation_engineered"])
for key, value in compare_group.items():
    print(f"{key}: {value}")

In [ ]:
raise

In [ ]:
thresholds = {
    "accuracy": 0.003,
    "f1": -0.01,
}

for model in MODEL_REGISTRY:

    result = recommended_by_domain_for_model(
        results_df=all_results,
        model_name=model,
        thresholds=thresholds,
    )

    print(f"\nBest candidates for {model}")
    print("=" * 50)

    print("\nRecommended:")
    for recommendation in result["recommended"]:
        print(
            recommendation["domain"],
            "->",
            recommendation["group"],
            recommendation["deltas"],
        )
    print()
    print("recommended list:")
    for recommendation in result["recommended"]:
        print(recommendation["group"], end=", ")
    print()
    print("\nFull domain results:")
    print(result["df"].to_markdown(index=False))

In [ ]:
# import pprint
# Feature_effect = analyze_feature_effect(workflow['comparison'])
# pprint.pprint(Feature_effect)

In [ ]:
# Reminder of how to run a single experiment if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# exp_config = [exp_config]
# exp_result = run_experiments(train_df, exp_config, target=TARGET,)
# exp_report = experiment_report(exp_result, exp_config, print_report=True)
# save_results(exp_result)
# save_configs(exp_config)

In [ ]:
# Summary for baseline experiment to use in report, 
# since we can't compare it to itself.

# baseline_summary = baseline_summary_to_markdown(exp_result)
# print("Baseline summary:")
# print(baseline_summary)

In [ ]:
# Reminder for how to load results and configs if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# all_results = load_results()
# print("Loaded results:")
# print(all_results)

# all_configs = load_configs()
# print("Loaded configs:")
# print(all_configs)

In [ ]:
# print(workflow["all_results"])

In [ ]:
# for model in MODEL_REGISTRY:
#     model_progression_df = model_progression(workflow["all_results"], model_name=model, metric="test_accuracy_mean")
#     print(f"Model progression for {model}:")
#     print(model_progression_df)
#     print()

Old Eda practice

In [ ]:
# eda = run_eda(train_df, target=TARGET, display=True, head=3,random=4, tail=3)
# print(eda)

In [ ]:
# print(eda)

In [ ]:
# print(eda['dataFrame_health'].to_markdown())

In [ ]:
# print(eda["dataFrame_summary"].to_markdown())

In [ ]:
# for col in eda['categorical_summary']:
#     sample = sample_dataframe(eda['categorical_summary'][col], head=1, random=3, tail=1)
#     sample_df = pd.concat(
#         [sample[k] for k in ['head', 'random', 'tail']],
#         ignore_index=False
#     )
#     print(sample_df.to_markdown())
#     print()

In [ ]:
# for col in eda['numerical_summary']:
#     print(f"Numerical column: {col}")
#     print(eda['numerical_summary'][col].to_markdown())
#     print()

In [ ]:
# print("Correlation matrix:")
# print(eda["correlation_matrix"].to_markdown())

In [ ]:
# print('Correlation with the target variable:')
# print(eda["target_correlation"].to_markdown())

In [ ]:
# for col in eda["categorical_rare"]:
#     print(f"Categorical column with rare values: {col}")
#     print(eda["categorical_rare"][col])
#     print()

In [ ]:
# # print(eda['sample'])
# sample_df = pd.concat(
#     [eda['sample'][k] for k in ['head', 'random', 'tail']],
#     ignore_index=True
# )
# print(sample_df.to_markdown(index=False))

In [ ]:

TARGET = "Survived"
ALL_EXPERIMENTS = ALL_EXPERIMENTS
for experiment_name, exp_config in ALL_EXPERIMENTS.items():
    print(f"Experiment: {experiment_name}")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

exp_configs = ALL_EXPERIMENTS["baseline__raw"]


Experiment: baseline__raw
Experiment: fe01__family
Experiment: fe02__has_cabin
Experiment: fe03__deck
Experiment: fe04__cabin_features
Experiment: fe05__title
Experiment: fe06__age_imputation_title
Experiment: fe07__age_imputation_title_pclass
Experiment: fe08__fare_per_family_member
Experiment: fe11__age_bin
Experiment: fe12__sex_pclass
Experiment: cb01__age_and_bins
Experiment: cb02__age_imputed_title_and_bins
Experiment: cb03__age_imputed_title_pclass_and_bins
Experiment: cb04__fare_and_fare_per_family
Experiment: cb07__family_features
Experiment: cb08__sex_pclass_features
Experiment: ab01__age_and_bins_without_fare
Experiment: ab02__age_imputed_title_and_bins_without_fare
Experiment: ab03__age_imputed_title_pclass_and_bins_without_fare
Experiment: ab04__age_bin_without_fare
Experiment: ab05__sex_pclass_without_sibsp_parch
Experiment: control__cb01_legacy_order
Experiment: fe09__ticket_group_size_batch
Experiment: fe09__ticket_group_size_fitted
Experiment: fe09__ticket_group_size_fu

#-----------------------------------------------------------------

In [17]:
from titanic_ml.common.experiments.runner import apply_pre_cv_feature_pipeline, build_model_pipeline

In [18]:
from titanic_ml.common.experiments.configurations.configuration import ALL_HPO_CONFIGS

In [19]:
from titanic_ml.common.experiments.hpo import run_hyperparameter_search

In [20]:
from titanic_ml.common.experiments.hpo import run_hyperparameter_search
from titanic_ml.common.experiments.report import summarize_hpo_results
searches, results = run_hyperparameter_search(
    df=train_df,
    experiments=ALL_HPO_CONFIGS['hpo06_configs__svc'],
    target="Survived",
    context_df=test_df
)

search = next(iter(searches.values()))

print("Best parameters:", search.best_params_)
print("Best CV accuracy:", search.best_score_)

display(summarize_hpo_results(results))

Best parameters: {'model__C': 1.0, 'model__class_weight': None, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
Best CV accuracy: 0.8361182599962337


,Experiment,C,class_weight,gamma,kernel,CV Accuracy,Accuracy Std,CV F1,F1 Std,Train Accuracy,Rank
0,hpo06__class_weight09__svc,1.0,None,scale,rbf,0.83612,0.02068,0.77437,0.03856,0.83922,1
1,hpo06__class_weight17__svc,1.5,None,scale,rbf,0.83612,0.02068,0.77437,0.03856,0.84091,1
2,hpo06__class_weight01__svc,0.6,None,scale,rbf,0.83499,0.01917,0.77316,0.03715,0.83782,3
3,hpo06__class_weight06__svc,0.6,"{0: 1.25, 1: 1}",scale,rbf,0.83499,0.02135,0.77190,0.03877,0.83782,3
4,hpo06__class_weight14__svc,1.0,"{0: 1.25, 1: 1}",scale,rbf,0.83499,0.01949,0.77129,0.03661,0.84203,3
5,hpo06__class_weight11__svc,1.0,"{0: 1, 1: 1.25}",scale,rbf,0.83275,0.01973,0.77078,0.03809,0.83726,6
6,hpo06__class_weight19__svc,1.5,"{0: 1, 1: 1.25}",scale,rbf,0.83163,0.02024,0.76963,0.03827,0.84035,7
7,hpo06__class_weight07__svc,0.6,"{0: 1.5, 1: 1}",scale,rbf,0.83052,0.01798,0.76176,0.03312,0.84287,8
8,hpo06__class_weight15__svc,1.0,"{0: 1.5, 1: 1}",scale,rbf,0.83052,0.02552,0.75237,0.04839,0.84849,8
9,hpo06__class_weight24__svc,1.5,"{0: 2, 1: 1}",scale,rbf,0.82719,0.02522,0.72851,0.05561,0.84259,10


In [ ]:
searches, results = run_hyperparameter_search(
    df=train_df,
    experiments=ALL_HPO_CONFIGS["hpo06_configs__svc"],
    target="Survived",
    context_df=test_df,
)

metadata_columns = [
    "experiment",
    "source_search",
    "candidate_id",
    "stage",
    "feature_group",
    "group",
    "model_name",
    "domain",
    "param_model__C",
    "param_model__class_weight",
    "rank_test_accuracy",
]

display(results[metadata_columns].head())

assert results["experiment"].is_unique
assert results["candidate_id"].nunique() == len(results)
assert results["source_search"].nunique() == 1

,experiment,source_search,candidate_id,stage,feature_group,group,model_name,domain,param_model__C,rank_test_accuracy
0,hpo01__regularization01__logreg,hpo01__regularization__logreg,1,hpo01,regularization,hpo01__regularization,logreg,logreg,0.01,5
1,hpo01__regularization02__logreg,hpo01__regularization__logreg,2,hpo01,regularization,hpo01__regularization,logreg,logreg,0.10,4
2,hpo01__regularization03__logreg,hpo01__regularization__logreg,3,hpo01,regularization,hpo01__regularization,logreg,logreg,1.00,1
3,hpo01__regularization04__logreg,hpo01__regularization__logreg,4,hpo01,regularization,hpo01__regularization,logreg,logreg,10.00,2
4,hpo01__regularization05__logreg,hpo01__regularization__logreg,5,hpo01,regularization,hpo01__regularization,logreg,logreg,100.00,3


In [22]:
display(
    results[
        [
            "experiment",
            "source_search",
            "stage",
            "feature_group",
            "group",
            "domain",
        ]
    ].head()
)

,experiment,source_search,stage,feature_group,group,domain
0,hpo06__class_weight01__svc,svc_hpo05__kernel_comparison,hpo06,class_weight,svc_hpo05__kernel_comparison,svc
1,hpo06__class_weight02__svc,svc_hpo05__kernel_comparison,hpo06,class_weight,svc_hpo05__kernel_comparison,svc
2,hpo06__class_weight03__svc,svc_hpo05__kernel_comparison,hpo06,class_weight,svc_hpo05__kernel_comparison,svc
3,hpo06__class_weight04__svc,svc_hpo05__kernel_comparison,hpo06,class_weight,svc_hpo05__kernel_comparison,svc
4,hpo06__class_weight05__svc,svc_hpo05__kernel_comparison,hpo06,class_weight,svc_hpo05__kernel_comparison,svc


In [23]:
searches, results = run_hyperparameter_search(
    df=train_df,
    experiments=ALL_HPO_CONFIGS["hpo03_configs__svc"],
    target="Survived",
    context_df=test_df
)

search = next(iter(searches.values()))

print(search.best_params_)
print(search.best_score_)

{'model__C': 1.0, 'model__gamma': 'scale'}
0.8361182599962337


In [24]:
print("Best parameters:", search.best_params_)
print("Best CV accuracy:", search.best_score_)

display(
    results[
        [
            "param_model__C",
            "mean_test_accuracy",
            "std_test_accuracy",
            "mean_test_f1",
            "std_test_f1",
            "mean_train_accuracy",
            "rank_test_accuracy",
        ]
    ].sort_values("rank_test_accuracy")
)

Best parameters: {'model__C': 1.0, 'model__gamma': 'scale'}
Best CV accuracy: 0.8361182599962337


,param_model__C,mean_test_accuracy,std_test_accuracy,mean_test_f1,std_test_f1,mean_train_accuracy,rank_test_accuracy
5,1.0,0.836118,0.020679,0.774372,0.038563,0.839225,1
10,1.5,0.836118,0.020679,0.774372,0.038563,0.840908,1
13,1.5,0.836118,0.020679,0.774372,0.038563,0.842030,1
0,0.6,0.834995,0.019168,0.773156,0.037146,0.837822,4
8,1.0,0.834995,0.021349,0.772469,0.039287,0.840908,4
15,2.0,0.834995,0.022218,0.773175,0.040526,0.842311,4
3,0.6,0.834995,0.019168,0.773156,0.037146,0.838664,4
18,2.0,0.829396,0.023525,0.760925,0.046207,0.846240,8
17,2.0,0.821524,0.029728,0.758341,0.050014,0.825757,9
4,0.6,0.814858,0.031068,0.749850,0.043599,0.889452,10


In [25]:
def summarize_hpo_results(results):
    columns = {
        "experiment": "Experiment",
        "param_model__C": "C",
        "mean_test_accuracy": "CV Accuracy",
        "std_test_accuracy": "Accuracy Std",
        "mean_test_f1": "CV F1",
        "std_test_f1": "F1 Std",
        "mean_train_accuracy": "Train Accuracy",
        "rank_test_accuracy": "Rank",
    }

    available = {
        key: value
        for key, value in columns.items()
        if key in results.columns
    }

    summary = results[list(available)].rename(
        columns=available
    )

    return summary.sort_values(
        ["CV Accuracy", "CV F1"],
        ascending=False,
    ).round(5)

In [26]:
summarize_hpo_results(results)

,Experiment,C,CV Accuracy,Accuracy Std,CV F1,F1 Std,Train Accuracy,Rank
5,hpo__fare_fe10_full_context06__svc,1.0,0.83612,0.02068,0.77437,0.03856,0.83922,1
10,hpo__fare_fe10_full_context11__svc,1.5,0.83612,0.02068,0.77437,0.03856,0.84091,1
13,hpo__fare_fe10_full_context14__svc,1.5,0.83612,0.02068,0.77437,0.03856,0.84203,1
15,hpo__fare_fe10_full_context16__svc,2.0,0.83499,0.02222,0.77318,0.04053,0.84231,4
0,hpo__fare_fe10_full_context01__svc,0.6,0.83499,0.01917,0.77316,0.03715,0.83782,4
3,hpo__fare_fe10_full_context04__svc,0.6,0.83499,0.01917,0.77316,0.03715,0.83866,4
8,hpo__fare_fe10_full_context09__svc,1.0,0.83499,0.02135,0.77247,0.03929,0.84091,4
18,hpo__fare_fe10_full_context19__svc,2.0,0.82940,0.02353,0.76093,0.04621,0.84624,8
17,hpo__fare_fe10_full_context18__svc,2.0,0.82152,0.02973,0.75834,0.05001,0.82576,9
4,hpo__fare_fe10_full_context05__svc,0.6,0.81486,0.03107,0.74985,0.04360,0.88945,10


In [27]:
def summarize_hpo_results(results, decimals=5):
    parameter_columns = [
        column
        for column in results.columns
        if column.startswith("param_")
    ]

    metric_columns = [
        "mean_test_accuracy",
        "std_test_accuracy",
        "mean_test_f1",
        "std_test_f1",
        "mean_train_accuracy",
        "rank_test_accuracy",
    ]

    selected_columns = [
        "experiment",
        *parameter_columns,
        *[
            column
            for column in metric_columns
            if column in results.columns
        ],
    ]

    summary = results[selected_columns].copy()

    summary = summary.rename(
        columns={
            "experiment": "Experiment",
            **{
                column: column.removeprefix("param_model__")
                for column in parameter_columns
            },
            "mean_test_accuracy": "CV Accuracy",
            "std_test_accuracy": "Accuracy Std",
            "mean_test_f1": "CV F1",
            "std_test_f1": "F1 Std",
            "mean_train_accuracy": "Train Accuracy",
            "rank_test_accuracy": "Rank",
        }
    )

    return (
        summary
        .sort_values(
            ["CV Accuracy", "CV F1"],
            ascending=False,
        )
        .round(decimals)
        .reset_index(drop=True)
    )

In [28]:
# quick breakpoint
raise

RuntimeError: No active exception to reraise

In [ ]:
searches, results = run_hyperparameter_search(
    df=train_df,
    experiments=ALL_HPO_CONFIGS['hpo04_configs__logreg'],
    target="Survived",
)

search = next(iter(searches.values()))

print("Best parameters:", search.best_params_)
print("Best CV accuracy:", search.best_score_)

display(summarize_hpo_results(results))

Best parameters: {'model__C': 0.8, 'model__penalty': 'l2', 'model__solver': 'newton-cg'}
Best CV accuracy: 0.8417487916640513


,Experiment,C,penalty,solver,CV Accuracy,Accuracy Std,CV F1,F1 Std,Train Accuracy,Rank
0,hpo04__solver_comparison__logreg,0.8,l2,newton-cg,0.84175,0.01567,0.78790,0.02976,0.84484,1
1,hpo04__solver_comparison__logreg,0.8,l2,sag,0.84175,0.01567,0.78790,0.02976,0.84484,1
2,hpo04__solver_comparison__logreg,0.8,l2,saga,0.84175,0.01567,0.78790,0.02976,0.84484,1
3,hpo04__solver_comparison__logreg,1.0,l2,lbfgs,0.84063,0.01570,0.78675,0.02949,0.84512,4
4,hpo04__solver_comparison__logreg,1.0,l2,newton-cg,0.84063,0.01570,0.78675,0.02949,0.84512,4
5,hpo04__solver_comparison__logreg,0.6,l2,lbfgs,0.84063,0.01361,0.78624,0.02709,0.84596,6
6,hpo04__solver_comparison__logreg,0.6,l2,newton-cg,0.84063,0.01361,0.78624,0.02709,0.84596,6
7,hpo04__solver_comparison__logreg,0.6,l2,sag,0.84063,0.01361,0.78624,0.02709,0.84596,6
8,hpo04__solver_comparison__logreg,0.6,l2,saga,0.84063,0.01361,0.78624,0.02709,0.84596,6
9,hpo04__solver_comparison__logreg,0.8,l2,lbfgs,0.84063,0.01361,0.78624,0.02709,0.84456,6


In [ ]:
import pandas as pd

def has_varying_values(series):
    """
    Return True if a Series contains different values.

    Supports scalar hyperparameters and dictionary values.
    Treats missing scalar values as equivalent.
    """
    if series.empty:
        return False

    first_value = series.iloc[0]

    for value in series.iloc[1:]:

        # Handle missing scalar values.
        first_missing = (
            first_value is None
            or first_value is pd.NA
            or (
                isinstance(first_value, float)
                and pd.isna(first_value)
            )
        )

        value_missing = (
            value is None
            or value is pd.NA
            or (
                isinstance(value, float)
                and pd.isna(value)
            )
        )

        if first_missing and value_missing:
            continue

        if first_missing != value_missing:
            return True

        if value != first_value:
            return True

    return False

def hpo_leaderboard(results, top_n=5, decimals=5):
    if top_n < 1:
        raise ValueError("top_n must be at least 1.")

    # Detect searched hyperparameters.
    param_columns = [
        col for col in results.columns
        if col.startswith("param_")
    ]

    # Keep only parameters that vary across candidates.
    varying_params = [
        col for col in param_columns
        if has_varying_values(results[col])
    ]

    columns = [
        "experiment",
        *varying_params,
        "mean_test_accuracy",
        "std_test_accuracy",
        "mean_test_f1",
        "std_test_f1",
        "rank_test_accuracy",
    ]

    leaderboard = results[columns].copy()

    leaderboard = leaderboard.sort_values(
        [
            "mean_test_accuracy",
            "mean_test_f1",
            "std_test_accuracy",
        ],
        ascending=[False, False, True],
        kind="stable",
    ).head(top_n)

    leaderboard = leaderboard.rename(
        columns={
            "experiment": "Experiment",
            **{
                col: col.removeprefix("param_model__")
                for col in varying_params
            },
            "mean_test_accuracy": "CV Accuracy",
            "std_test_accuracy": "Accuracy Std",
            "mean_test_f1": "CV F1",
            "std_test_f1": "F1 Std",
            "rank_test_accuracy": "Rank",
        }
    )

    return leaderboard.round(decimals).reset_index(drop=True)

In [ ]:
display(hpo_leaderboard(results))

,Experiment,C,solver,CV Accuracy,Accuracy Std,CV F1,F1 Std,Rank
0,hpo04__solver_comparison__logreg,0.8,newton-cg,0.84175,0.01567,0.78790,0.02976,1
1,hpo04__solver_comparison__logreg,0.8,sag,0.84175,0.01567,0.78790,0.02976,1
2,hpo04__solver_comparison__logreg,0.8,saga,0.84175,0.01567,0.78790,0.02976,1
3,hpo04__solver_comparison__logreg,1.0,lbfgs,0.84063,0.01570,0.78675,0.02949,4
4,hpo04__solver_comparison__logreg,1.0,newton-cg,0.84063,0.01570,0.78675,0.02949,4


In [ ]:
searches, results = run_hyperparameter_search(
    df=train_df,
    experiments=ALL_HPO_CONFIGS['hpo06_configs__svc'],
    target="Survived",
    context_df=test_df
)

search = next(iter(searches.values()))

print("Best parameters:", search.best_params_)
print("Best CV accuracy:", search.best_score_)

display(summarize_hpo_results(results))

display(hpo_leaderboard(results))

Best parameters: {'model__C': 1.0, 'model__class_weight': None, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
Best CV accuracy: 0.8361182599962337


,Experiment,C,class_weight,gamma,kernel,CV Accuracy,Accuracy Std,CV F1,F1 Std,Train Accuracy,Rank
0,svc_hpo06__class_weight,1.0,None,scale,rbf,0.83612,0.02068,0.77437,0.03856,0.83922,1
1,svc_hpo06__class_weight,1.5,None,scale,rbf,0.83612,0.02068,0.77437,0.03856,0.84091,1
2,svc_hpo06__class_weight,0.6,None,scale,rbf,0.83499,0.01917,0.77316,0.03715,0.83782,3
3,svc_hpo06__class_weight,0.6,"{0: 1.25, 1: 1}",scale,rbf,0.83499,0.02135,0.77190,0.03877,0.83782,3
4,svc_hpo06__class_weight,1.0,"{0: 1.25, 1: 1}",scale,rbf,0.83499,0.01949,0.77129,0.03661,0.84203,3
5,svc_hpo06__class_weight,1.0,"{0: 1, 1: 1.25}",scale,rbf,0.83275,0.01973,0.77078,0.03809,0.83726,6
6,svc_hpo06__class_weight,1.5,"{0: 1, 1: 1.25}",scale,rbf,0.83163,0.02024,0.76963,0.03827,0.84035,7
7,svc_hpo06__class_weight,0.6,"{0: 1.5, 1: 1}",scale,rbf,0.83052,0.01798,0.76176,0.03312,0.84287,8
8,svc_hpo06__class_weight,1.0,"{0: 1.5, 1: 1}",scale,rbf,0.83052,0.02552,0.75237,0.04839,0.84849,8
9,svc_hpo06__class_weight,1.5,"{0: 2, 1: 1}",scale,rbf,0.82719,0.02522,0.72851,0.05561,0.84259,10


,Experiment,C,class_weight,CV Accuracy,Accuracy Std,CV F1,F1 Std,Rank
0,svc_hpo06__class_weight,1.0,None,0.83612,0.02068,0.77437,0.03856,1
1,svc_hpo06__class_weight,1.5,None,0.83612,0.02068,0.77437,0.03856,1
2,svc_hpo06__class_weight,0.6,None,0.83499,0.01917,0.77316,0.03715,3
3,svc_hpo06__class_weight,0.6,"{0: 1.25, 1: 1}",0.83499,0.02135,0.77190,0.03877,3
4,svc_hpo06__class_weight,1.0,"{0: 1.25, 1: 1}",0.83499,0.01949,0.77129,0.03661,3


In [ ]:
comparison = results.loc[
    results["param_model__C"] == 1.0,
    [
        "param_model__class_weight",
        "mean_test_accuracy",
        "mean_test_precision",
        "mean_test_recall",
        "mean_test_f1",
    ],
].copy()

comparison = comparison.rename(columns={
    "param_model__class_weight": "Class Weight",
    "mean_test_accuracy": "Accuracy",
    "mean_test_precision": "Precision",
    "mean_test_recall": "Recall",
    "mean_test_f1": "F1",
})

display(comparison.round(5).reset_index(drop=True))

,Class Weight,Accuracy,Precision,Recall,F1
0,None,0.83612,0.81706,0.73943,0.77437
1,balanced,0.81484,0.76835,0.75115,0.75572
2,"{0: 1, 1: 1.25}",0.83275,0.80879,0.73943,0.77078
3,"{0: 1, 1: 1.5}",0.82267,0.78516,0.74527,0.76189
4,"{0: 1, 1: 2}",0.79686,0.71759,0.78905,0.74731
5,"{0: 1.25, 1: 1}",0.83499,0.82034,0.73069,0.77129
6,"{0: 1.5, 1: 1}",0.83052,0.84963,0.67809,0.75237
7,"{0: 2, 1: 1}",0.82270,0.90671,0.60239,0.72013


In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
def select_best_threshold(y_true, probabilities, thresholds):
    scores = []

    for threshold in thresholds:
        predictions = (probabilities >= threshold).astype(int)

        scores.append({
            "threshold": threshold,
            "accuracy": accuracy_score(y_true, predictions),
            "precision": precision_score(
                y_true, predictions, zero_division=0
            ),
            "recall": recall_score(
                y_true, predictions, zero_division=0
            ),
            "f1": f1_score(
                y_true, predictions, zero_division=0
            ),
        })

    results = pd.DataFrame(scores)

    results["distance_from_default"] = abs(
        results["threshold"] - 0.5
    )

    ranked = results.sort_values(
        ["accuracy", "f1", "distance_from_default"],
        ascending=[False, False, True],
        kind="stable",
    )

    best_threshold = float(ranked.iloc[0]["threshold"])

    return best_threshold, results

In [ ]:
def evaluate_threshold_cv(
    X,
    y,
    pipeline,
    thresholds,
    outer_cv=None,
    inner_cv=None,
):

    if outer_cv is None:
        outer_cv = StratifiedKFold(n_splits=5)

    rows = []
    candidate_results = []

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y), start=1
    ):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        # Inner CV: generate probabilities for threshold selection.
        if inner_cv is None:
            inner_cv = StratifiedKFold(n_splits=5,)

        inner_probabilities = cross_val_predict(
            clone(pipeline),
            X_train,
            y_train,
            cv=inner_cv,
            method="predict_proba",
            n_jobs=-1,
        )[:, 1]

        best_threshold, fold_candidates = select_best_threshold(
            y_train,
            inner_probabilities,
            thresholds,
        )

        fold_candidates.insert(0, "outer_fold", fold)
        fold_candidates["selected"] = (
            fold_candidates["threshold"] == best_threshold
        )

        candidate_results.append(fold_candidates)

        # Fit using all outer training observations.
        model = clone(pipeline)
        model.fit(X_train, y_train)

        # Predict on the untouched outer validation fold.
        probabilities = model.predict_proba(X_test)[:, 1]

        for name, threshold in [
            ("default", 0.5),
            ("tuned", best_threshold),
        ]:
            predictions = (probabilities >= threshold).astype(int)

            rows.append({
                "fold": fold,
                "method": name,
                "threshold": threshold,
                "accuracy": accuracy_score(y_test, predictions),
                "precision": precision_score(
                    y_test, predictions, zero_division=0
                ),
                "recall": recall_score(
                    y_test, predictions, zero_division=0
                ),
                "f1": f1_score(
                    y_test, predictions, zero_division=0
                ),
            })

    return (
        pd.DataFrame(rows),
        pd.concat(candidate_results, ignore_index=True),
    )

In [ ]:
def select_best_threshold(
    y_true,
    scores,
    thresholds,
    default_threshold=0.5,
):
    results = []

    thresholds = np.asarray(thresholds)

    if thresholds.ndim != 1 or thresholds.size == 0:
        raise ValueError(
            "thresholds must be a non-empty one-dimensional collection."
        )

    for threshold in thresholds:
        predictions = (scores >= threshold).astype(int)

        results.append({
            "threshold": threshold,
            "accuracy": accuracy_score(y_true, predictions),
            "precision": precision_score(
                y_true, predictions, zero_division=0
            ),
            "recall": recall_score(
                y_true, predictions, zero_division=0
            ),
            "f1": f1_score(
                y_true, predictions, zero_division=0
            ),
        })

    results = pd.DataFrame(results)

    results["distance_from_default"] = abs(
        results["threshold"] - default_threshold
    )

    ranked = results.sort_values(
        ["accuracy", "f1", "distance_from_default"],
        ascending=[False, False, True],
        kind="stable",
    )

    return float(ranked.iloc[0]["threshold"]), results


def evaluate_threshold_cv(
    X,
    y,
    pipeline,
    thresholds,
    outer_cv=None,
    inner_cv=None,
    score_method="predict_proba",
    default_threshold=None,
):
    if score_method not in ("predict_proba", "decision_function"):
        raise ValueError(
            "score_method must be 'predict_proba' or 'decision_function'."
        )

    if outer_cv is None:
        outer_cv = StratifiedKFold(n_splits=5)

    if inner_cv is None:
        inner_cv = StratifiedKFold(n_splits=5)

    if default_threshold is None:
        default_threshold = (
            0.5 if score_method == "predict_proba" else 0.0
        )

    rows = []
    candidate_results = []

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y), start=1
    ):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        # Inner CV: generate out-of-fold scores for threshold selection.
        inner_scores = cross_val_predict(
            clone(pipeline),
            X_train,
            y_train,
            cv=inner_cv,
            method=score_method,
            n_jobs=-1,
        )

        if score_method == "predict_proba":
            inner_scores = inner_scores[:, 1]

        best_threshold, fold_candidates = select_best_threshold(
            y_train,
            inner_scores,
            thresholds,
            default_threshold=default_threshold,
        )

        fold_candidates.insert(0, "outer_fold", fold)
        fold_candidates["selected"] = (
            fold_candidates["threshold"] == best_threshold
        )

        candidate_results.append(fold_candidates)

        # Fit using all outer training observations.
        model = clone(pipeline)
        model.fit(X_train, y_train)

        # Generate scores for the untouched outer validation fold.
        if score_method == "predict_proba":
            outer_scores = model.predict_proba(X_test)[:, 1]
        else:
            outer_scores = model.decision_function(X_test)

        for name, threshold in [
            ("default", default_threshold),
            ("tuned", best_threshold),
        ]:
            predictions = (outer_scores >= threshold).astype(int)

            rows.append({
                "fold": fold,
                "method": name,
                "threshold": threshold,
                "accuracy": accuracy_score(y_test, predictions),
                "precision": precision_score(
                    y_test, predictions, zero_division=0
                ),
                "recall": recall_score(
                    y_test, predictions, zero_division=0
                ),
                "f1": f1_score(
                    y_test, predictions, zero_division=0
                ),
            })

    return (
        pd.DataFrame(rows),
        pd.concat(candidate_results, ignore_index=True),
    )


In [ ]:
exp = next(iter(ALL_FS_CONFIGS["pr03__removing_fare__logreg"].values()))
pipeline = build_model_pipeline(exp)

thresholds = np.round(
    np.arange(0.30, 0.701, 0.025),
    3,
)


X = train_df.drop(columns=["Survived"])
y = train_df["Survived"]

hpo06_results, hpo06_candidates = evaluate_threshold_cv(
    X=X,
    y=y,
    pipeline=pipeline,
    thresholds=thresholds,
)

display(hpo06_results)
display(hpo06_candidates)

,fold,method,threshold,accuracy,precision,recall,f1
0,1,default,0.500,0.837989,0.785714,0.797101,0.791367
1,1,tuned,0.500,0.837989,0.785714,0.797101,0.791367
2,2,default,0.500,0.831461,0.806452,0.735294,0.769231
3,2,tuned,0.500,0.831461,0.806452,0.735294,0.769231
4,3,default,0.500,0.837079,0.782609,0.794118,0.788321
5,3,tuned,0.500,0.837079,0.782609,0.794118,0.788321
6,4,default,0.500,0.825843,0.836364,0.676471,0.747967
7,4,tuned,0.500,0.825843,0.836364,0.676471,0.747967
8,5,default,0.500,0.870787,0.819444,0.855072,0.836879
9,5,tuned,0.525,0.865169,0.826087,0.826087,0.826087


,outer_fold,threshold,accuracy,precision,recall,f1,distance_from_default,selected
0,1,0.300,0.790730,0.691358,0.820513,0.750419,0.200,False
1,1,0.325,0.799157,0.707006,0.813187,0.756388,0.175,False
2,1,0.350,0.806180,0.718447,0.813187,0.762887,0.150,False
3,1,0.375,0.811798,0.729373,0.809524,0.767361,0.125,False
4,1,0.400,0.816011,0.739865,0.802198,0.769772,0.100,False
...,...,...,...,...,...,...,...,...
80,5,0.600,0.812062,0.808889,0.666667,0.730924,0.100,False
81,5,0.625,0.802244,0.811321,0.630037,0.709278,0.125,False
82,5,0.650,0.800842,0.822660,0.611722,0.701681,0.150,False
83,5,0.675,0.791024,0.840659,0.560440,0.672527,0.175,False


In [ ]:
threshold_summary = (
    hpo06_candidates
    .groupby("threshold")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean"),
        selected_count=("selected", "sum"),
    )
    .round(5)
    .reset_index()
)

display(threshold_summary)

,threshold,accuracy_mean,accuracy_std,precision_mean,recall_mean,f1_mean,selected_count
0,0.300,0.79321,0.01258,0.69476,0.82309,0.75348,0
1,0.325,0.80107,0.00670,0.70894,0.81724,0.75924,0
2,0.350,0.80415,0.00581,0.71752,0.80774,0.75994,0
3,0.375,0.80920,0.00955,0.72904,0.80043,0.76303,0
4,0.400,0.81482,0.00996,0.74185,0.79385,0.76692,0
5,0.425,0.81706,0.01048,0.75081,0.78361,0.76680,0
6,0.450,0.82155,0.00809,0.76206,0.77849,0.77007,0
7,0.475,0.82688,0.00717,0.77885,0.76753,0.77297,0
8,0.500,0.83193,0.00350,0.79478,0.75803,0.77589,4
9,0.525,0.83193,0.00321,0.80215,0.74634,0.77319,1


In [ ]:
thresholds_by_fold = hpo06_results.loc[
    hpo06_results["method"] == "tuned",
    ["fold", "threshold"],
]

performance = (
    hpo06_results
    .groupby("method")[
        ["accuracy", "precision", "recall", "f1"]
    ]
    .agg(["mean", "std"])
    .round(5)
)

display(thresholds_by_fold)
display(performance)

,fold,threshold
1,1,0.500
3,2,0.500
5,3,0.500
7,4,0.500
9,5,0.525


accuracy          precision            recall                f1  \
            mean      std      mean      std     mean      std     mean   
method                                                                    
default  0.84063  0.01755   0.80612  0.02270  0.77161  0.06799  0.78675   
tuned    0.83951  0.01515   0.80745  0.02384  0.76581  0.05985  0.78459   

                  
             std  
method            
default  0.03297  
tuned    0.02898

In [ ]:

# Retrieve the selected SVC feature configuration.
exp = next(iter(
    ALL_FS_CONFIGS["fs04__fare_fe10_full_context__svc"].values()
))

# Prepare the original predictors and target.
X = train_df.drop(columns=["Survived"])
y = train_df["Survived"]

# Apply the same pre-CV feature engineering used in HPO01-HPO06.
X = apply_pre_cv_feature_pipeline(
    X=X,
    exp=exp,
    context_df=test_df,
)

# Build the model pipeline.
pipeline = build_model_pipeline(exp)

# Candidate decision thresholds.
thresholds = np.round(
    np.arange(-1.0, 1.01, 0.1),
    2,
)

# Evaluate thresholds using nested CV.
hpo07_results, hpo07_candidates = evaluate_threshold_cv(
    X=X,
    y=y,
    pipeline=pipeline,
    thresholds=thresholds,
    score_method="decision_function",
    default_threshold=0.0,
)

display(hpo07_results)
display(hpo07_candidates)


,fold,method,threshold,accuracy,precision,recall,f1
0,1,default,0.0,0.854749,0.820896,0.797101,0.808824
1,1,tuned,-0.0,0.854749,0.820896,0.797101,0.808824
2,2,default,0.0,0.825843,0.784615,0.750000,0.766917
3,2,tuned,-0.0,0.825843,0.784615,0.750000,0.766917
4,3,default,0.0,0.825843,0.793651,0.735294,0.763359
5,3,tuned,-0.0,0.825843,0.793651,0.735294,0.763359
6,4,default,0.0,0.808989,0.840000,0.617647,0.711864
7,4,tuned,0.2,0.808989,0.840000,0.617647,0.711864
8,5,default,0.0,0.865169,0.846154,0.797101,0.820896
9,5,tuned,-0.0,0.865169,0.846154,0.797101,0.820896


,outer_fold,threshold,accuracy,precision,recall,f1,distance_from_default,selected
0,1,-1.0,0.698034,0.569378,0.871795,0.688857,1.0,False
1,1,-0.9,0.789326,0.700326,0.787546,0.741379,0.9,False
2,1,-0.8,0.800562,0.729825,0.761905,0.745520,0.8,False
3,1,-0.7,0.811798,0.750903,0.761905,0.756364,0.7,False
4,1,-0.6,0.816011,0.766917,0.747253,0.756957,0.6,False
...,...,...,...,...,...,...,...,...
100,5,0.6,0.809257,0.807175,0.659341,0.725806,0.6,False
101,5,0.7,0.805049,0.828431,0.619048,0.708595,0.7,False
102,5,0.8,0.798036,0.872832,0.553114,0.677130,0.8,False
103,5,0.9,0.788219,0.896104,0.505495,0.646370,0.9,False


In [ ]:
threshold_summary = (
    hpo07_candidates
    .groupby("threshold")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean"),
        selected_count=("selected", "sum"),
    )
    .round(5)
    .reset_index()
)

display(threshold_summary)

,threshold,accuracy_mean,accuracy_std,precision_mean,recall_mean,f1_mean,selected_count
0,-1.0,0.70006,0.01533,0.57308,0.85818,0.68717,0
1,-0.9,0.78956,0.00977,0.70569,0.77558,0.73888,0
2,-0.8,0.80079,0.01054,0.73210,0.75876,0.74518,0
3,-0.7,0.80836,0.00893,0.74730,0.75657,0.75189,0
4,-0.6,0.81257,0.00920,0.75927,0.74925,0.75419,0
5,-0.5,0.81987,0.00653,0.77627,0.74560,0.76054,0
6,-0.4,0.82351,0.00392,0.78581,0.74267,0.76354,0
7,-0.3,0.82800,0.00369,0.79616,0.74194,0.76798,0
8,-0.2,0.82912,0.00416,0.80011,0.73975,0.76862,0
9,-0.1,0.83165,0.00471,0.80676,0.73829,0.77088,0


In [ ]:
import numpy as np

thresholds = np.round(
    np.arange(-1.0, 1.01, 0.1),
    2,
)

In [ ]:
svc_model = searches["svc_hpo06__class_weight"].best_estimator_

scores = svc_model.decision_function(X)

display(
    pd.Series(scores).describe(
        percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
    )
)

count    891.000000
mean      -0.303182
std        0.986655
min       -1.701416
5%        -1.117643
25%       -1.001909
50%       -0.999603
75%        0.908141
95%        1.341693
max        1.580521
dtype: float64

In [ ]:
exp = next(iter(ALL_FS_CONFIGS["fs04__fare_fe10_full_context__svc"].values()))
pipeline = build_model_pipeline(exp)

thresholds = np.round(
    np.arange(-1.0, 1.01, 0.1),
    2,
)


X = train_df.drop(columns=["Survived"])
y = train_df["Survived"]

hpo06_results, hpo06_candidates = evaluate_threshold_cv(
    X=X,
    y=y,
    pipeline=pipeline,
    thresholds=thresholds,
    score_method="decision_function",
)

display(hpo06_results)
display(hpo06_candidates)

ValueError: Denominator column 'TicketGroupSize' not found.

In [ ]:
default_predictions = scores >= 0.0

threshold_changes = pd.DataFrame({
    "threshold": thresholds,
    "changed_predictions": [
        np.sum(
            (scores >= threshold) != default_predictions
        )
        for threshold in thresholds
    ],
})

display(threshold_changes)

,threshold,changed_predictions
0,-1.0,238
1,-0.9,35
2,-0.8,25
3,-0.7,19
4,-0.6,12
5,-0.5,7
6,-0.4,6
7,-0.3,4
8,-0.2,3
9,-0.1,1


In [ ]:
def select_best_threshold(
    y_true,
    scores,
    thresholds,
    default_threshold=0.5,
):
    results = []

    for threshold in thresholds:
        predictions = (scores >= threshold).astype(int)

        results.append({
            "threshold": threshold,
            "accuracy": accuracy_score(y_true, predictions),
            "precision": precision_score(
                y_true, predictions, zero_division=0
            ),
            "recall": recall_score(
                y_true, predictions, zero_division=0
            ),
            "f1": f1_score(
                y_true, predictions, zero_division=0
            ),
        })

    results = pd.DataFrame(results)

    results["distance_from_default"] = abs(
        results["threshold"] - default_threshold
    )

    ranked = results.sort_values(
        ["accuracy", "f1", "distance_from_default"],
        ascending=[False, False, True],
        kind="stable",
    )

    return float(ranked.iloc[0]["threshold"]), results


def evaluate_threshold_cv(
    X,
    y,
    thresholds,
    pipeline=None,
    exp=None,
    context_df=None,
    outer_cv=None,
    inner_cv=None,
    score_method="predict_proba",
    default_threshold=None,
):
    if score_method not in ("predict_proba", "decision_function"):
        raise ValueError(
            "score_method must be 'predict_proba' or 'decision_function'."
        )

    # -------------------------------------------------
    # 1. Resolve model pipeline and feature preparation
    # -------------------------------------------------

    # Checks both cases where both or none are provided
    if (exp is None) == (pipeline is None):
        raise ValueError(
            "Provide exactly one of 'exp' or 'pipeline'."
        )

    if exp is not None:
        if exp.get("pre_cv_feature_pipeline"):
            X = apply_pre_cv_feature_pipeline(
                X=X,
                exp=exp,
                context_df=context_df,
            )

        pipeline = build_model_pipeline(exp)

    elif context_df is not None:
        raise ValueError(
            "context_df requires an experiment configuration."
        )

    if outer_cv is None:
        outer_cv = StratifiedKFold(n_splits=5)

    if inner_cv is None:
        inner_cv = StratifiedKFold(n_splits=5)

    if default_threshold is None:
        default_threshold = (
            0.5 if score_method == "predict_proba" else 0.0
        )

    rows = []
    candidate_results = []


    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y), start=1
    ):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        # Inner CV: generate out-of-fold scores for threshold selection.
        inner_scores = cross_val_predict(
            clone(pipeline),
            X_train,
            y_train,
            cv=inner_cv,
            method=score_method,
            n_jobs=-1,
        )

        if score_method == "predict_proba":
            inner_scores = inner_scores[:, 1]

        best_threshold, fold_candidates = select_best_threshold(
            y_train,
            inner_scores,
            thresholds,
            default_threshold=default_threshold,
        )

        fold_candidates.insert(0, "outer_fold", fold)
        fold_candidates["selected"] = (
            fold_candidates["threshold"] == best_threshold
        )

        candidate_results.append(fold_candidates)

        # Fit using all outer training observations.
        model = clone(pipeline)
        model.fit(X_train, y_train)

        # Generate scores for the untouched outer validation fold.
        if score_method == "predict_proba":
            outer_scores = model.predict_proba(X_test)[:, 1]
        else:
            outer_scores = model.decision_function(X_test)

        for name, threshold in [
            ("default", default_threshold),
            ("tuned", best_threshold),
        ]:
            predictions = (outer_scores >= threshold).astype(int)

            rows.append({
                "fold": fold,
                "method": name,
                "threshold": threshold,
                "accuracy": accuracy_score(y_test, predictions),
                "precision": precision_score(
                    y_test, predictions, zero_division=0
                ),
                "recall": recall_score(
                    y_test, predictions, zero_division=0
                ),
                "f1": f1_score(
                    y_test, predictions, zero_division=0
                ),
            })

    return (
        pd.DataFrame(rows),
        pd.concat(candidate_results, ignore_index=True),
    )


In [ ]:

# Retrieve the selected SVC feature configuration.
exp = next(iter(
    ALL_FS_CONFIGS["fs04__fare_fe10_full_context__svc"].values()
))

# Prepare the original predictors and target.
X = train_df.drop(columns=["Survived"])
y = train_df["Survived"]

# Candidate decision thresholds.
thresholds = np.round(
    np.arange(-1.0, 1.01, 0.1),
    2,
)

# Evaluate thresholds using nested CV.
hpo07_results, hpo07_candidates = evaluate_threshold_cv(
    X=X,
    y=y,
    thresholds=thresholds,
    score_method="decision_function",
    default_threshold=0.0,
    exp=exp,
    context_df=test_df
)

display(hpo07_results)
display(hpo07_candidates)


,fold,method,threshold,accuracy,precision,recall,f1
0,1,default,0.0,0.854749,0.820896,0.797101,0.808824
1,1,tuned,-0.0,0.854749,0.820896,0.797101,0.808824
2,2,default,0.0,0.825843,0.784615,0.750000,0.766917
3,2,tuned,-0.0,0.825843,0.784615,0.750000,0.766917
4,3,default,0.0,0.825843,0.793651,0.735294,0.763359
5,3,tuned,-0.0,0.825843,0.793651,0.735294,0.763359
6,4,default,0.0,0.808989,0.840000,0.617647,0.711864
7,4,tuned,0.2,0.808989,0.840000,0.617647,0.711864
8,5,default,0.0,0.865169,0.846154,0.797101,0.820896
9,5,tuned,-0.0,0.865169,0.846154,0.797101,0.820896


,outer_fold,threshold,accuracy,precision,recall,f1,distance_from_default,selected
0,1,-1.0,0.698034,0.569378,0.871795,0.688857,1.0,False
1,1,-0.9,0.789326,0.700326,0.787546,0.741379,0.9,False
2,1,-0.8,0.800562,0.729825,0.761905,0.745520,0.8,False
3,1,-0.7,0.811798,0.750903,0.761905,0.756364,0.7,False
4,1,-0.6,0.816011,0.766917,0.747253,0.756957,0.6,False
...,...,...,...,...,...,...,...,...
100,5,0.6,0.809257,0.807175,0.659341,0.725806,0.6,False
101,5,0.7,0.805049,0.828431,0.619048,0.708595,0.7,False
102,5,0.8,0.798036,0.872832,0.553114,0.677130,0.8,False
103,5,0.9,0.788219,0.896104,0.505495,0.646370,0.9,False


In [ ]:
threshold_summary = (
    hpo07_candidates
    .groupby("threshold")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean"),
        selected_count=("selected", "sum"),
    )
    .round(5)
    .reset_index()
)

display(threshold_summary)

,threshold,accuracy_mean,accuracy_std,precision_mean,recall_mean,f1_mean,selected_count
0,-1.0,0.70006,0.01533,0.57308,0.85818,0.68717,0
1,-0.9,0.78956,0.00977,0.70569,0.77558,0.73888,0
2,-0.8,0.80079,0.01054,0.73210,0.75876,0.74518,0
3,-0.7,0.80836,0.00893,0.74730,0.75657,0.75189,0
4,-0.6,0.81257,0.00920,0.75927,0.74925,0.75419,0
5,-0.5,0.81987,0.00653,0.77627,0.74560,0.76054,0
6,-0.4,0.82351,0.00392,0.78581,0.74267,0.76354,0
7,-0.3,0.82800,0.00369,0.79616,0.74194,0.76798,0
8,-0.2,0.82912,0.00416,0.80011,0.73975,0.76862,0
9,-0.1,0.83165,0.00471,0.80676,0.73829,0.77088,0
